# Tutorial 05: DP-SGD for LoRA Fine-Tuning with HuggingFace

**Level**: Advanced
**Duration**: 60-90 minutes
**Prerequisites**: Tutorials 01-04, Basic familiarity with transformers and LoRA

---

## Overview

In this tutorial, we apply everything we've learned to a real-world use case: **fine-tuning large language models (LLMs) with differential privacy**.

**What you've learned so far**:
- **Tutorial 01**: Gradient clipping with `clipped_grad()`
- **Tutorial 02**: Noise injection and privacy accounting
- **Tutorial 03**: Complete DP-SGD training loop (manual SGD)

**What you'll learn today**:
1. Why LoRA (Low-Rank Adaptation) is ideal for DP fine-tuning
2. How to apply LoRA to HuggingFace models with PEFT
3. **Two ways to implement DP-AdamW**: Wrapper functions vs Opaque's optimizer API
4. Using PyTorch DataLoader for proper batch handling
5. Training with fixed batch sampling (not Poisson - that's Tutorial 05!)
6. Proper privacy accounting for fixed-size batches
7. Practical hyperparameter guidance for stable DP-SGD training

**Why LoRA + DP-SGD?**
- **Memory efficiency**: Only tune ~0.1% of parameters (millions vs billions)
- **Better privacy-utility**: Smaller gradients are easier to clip effectively
- **Faster convergence**: Less noise impact on small parameter space
- **Production ready**: HuggingFace integration makes deployment easy

**Tutorial Progression**:
- **Tutorial 03**: Manual batching + manual SGD (educational)
- **Tutorial 04** (this): DataLoader + DP-AdamW optimizer (production)
- **Tutorial 05** (next): Poisson sampling + microbatching

---

In [1]:
# Imports
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Opaque imports
from opaque import (
  make_functional,
  clipped_grad,
)

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print("All imports successful!")

W1113 15:55:24.881000 87006 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch version: 2.9.0
All imports successful!


---

## Part 1: Understanding LoRA

**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning method that adds trainable low-rank matrices to transformer layers.

### Standard Fine-Tuning vs LoRA

**Standard Fine-Tuning**:
```
Pre-trained weight: W ∈ ℝ^(d×k)
Update: W' = W + ΔW
Parameters to train: d × k (all of them!)
```

**LoRA Fine-Tuning**:
```
Pre-trained weight: W ∈ ℝ^(d×k) (frozen)
Low-rank update: ΔW = B·A where B ∈ ℝ^(d×r), A ∈ ℝ^(r×k)
Parameters to train: d·r + r·k (much smaller when r << min(d,k))
Scaling: ΔW is scaled by α/r before adding to W
```

**Example**:
- GPT-2: ~124M parameters total
- With LoRA (r=8, target only attention): ~300K trainable parameters (~0.24%)
- **500x fewer parameters to train!**

**Why LoRA is Perfect for DP-SGD**:

1. **Smaller gradients**: Fewer parameters = smaller gradient norms
   - Typical LoRA gradients: 0.1-2.0
   - Full fine-tuning gradients: 10-100+
   - **Clipping at appropriate scale preserves more signal**

2. **Better signal-to-noise ratio**: Less noise needed for same privacy
   - Noise scales with clip norm: σ = noise_multiplier × clip_norm
   - Smaller clip_norm → less noise → better learning

3. **Memory efficient**: Can fit larger models and batches in memory
   - Larger batches → better privacy-utility tradeoff

4. **Faster convergence**: DP noise has less impact on smaller parameter space

---

## Part 2: Setting Up the Model and Data

Let's load a pre-trained model and apply LoRA.

In [2]:
# Load a small model for demonstration (GPT-2 base)
model_name = "gpt2"
print(f"Loading {model_name}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"\nModel: {model_name}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading gpt2...

Model: gpt2
Total parameters: 124,439,808


### Apply LoRA with PEFT

We'll use the PEFT library to add LoRA adapters to the model.

In [3]:
# Configure LoRA
lora_config = LoraConfig(
  task_type=TaskType.CAUSAL_LM,
  r=8,  # Low-rank dimension (typical: 4-16)
  lora_alpha=16,  # Scaling factor (typical: 2×r for stable training)
  lora_dropout=0.0,  # Disable for deterministic training
  target_modules=["c_attn"],  # GPT-2: only attention layers
  init_lora_weights=True,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\n✓ LoRA applied successfully!")

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364

✓ LoRA applied successfully!


/Users/evgri243/Workspaces/external/opaque/.venv/lib/python3.11/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


**Key LoRA Hyperparameters**:

- **`r` (rank)**: Controls capacity and number of trainable parameters
  - Larger r = more capacity, more parameters, larger gradients
  - Typical: 4-16 for most tasks
  - **For DP**: Keep r small (4-8) for better gradient signal

- **`lora_alpha` (scaling)**: Controls contribution of LoRA to original weights
  - Final scaling = `alpha / r`
  - Typical: `alpha = 2×r` (scaling = 2.0)
  - **For DP**: Standard scaling (alpha=16 for r=8) works well

- **`target_modules`**: Which layers get LoRA
  - More modules = more parameters = larger gradients
  - GPT-2: `["c_attn"]` (attention only) is often sufficient
  - Can also target `["c_attn", "c_proj"]` for more capacity

---

## Part 3: Loading and Preparing Data

We'll use the AG News dataset for text classification.

In [18]:
# Training configuration
# These hyperparameters are from examples/gpt2_basic_training.py
# which has been tuned for stable DP-SGD + LoRA training
num_steps = 40
batch_size = 4
learning_rate = 5e-4  # Higher than non-DP (1e-4) to overcome noise

# Gradient clipping (fixed)
# Typical LoRA gradients are 0.1-2.0, so we clip at 0.1
clip_norm = 0.1

# Privacy parameters
# Lower noise_multiplier (0.5 vs 1.1) improves learning but reduces privacy
# Trade-off: Better utility vs stronger privacy guarantees
noise_multiplier = 0.5
sample_rate = 0.01  # For fixed batch: batch_size / dataset_size
target_delta = 1e-5

print(f"Training Configuration:")
print(f"  Steps: {num_steps}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Gradient clip norm: {clip_norm}")
print(f"\nPrivacy Configuration:")
print(f"  Noise multiplier: {noise_multiplier}")
print(f"  Sample rate: {sample_rate}")
print(f"  Target δ: {target_delta}")
print(f"\nNote: These hyperparameters are from examples/gpt2_basic_training.py")

Training Configuration:
  Steps: 40
  Batch size: 4
  Learning rate: 0.0005
  Gradient clip norm: 0.1

Privacy Configuration:
  Noise multiplier: 0.5
  Sample rate: 0.01
  Target δ: 1e-05

Note: These hyperparameters are from examples/gpt2_basic_training.py


In [19]:
# Load dataset
print("Loading AG News dataset...")
dataset = load_dataset("ag_news", split="train")

# Load only what we need for training
# With batch_size=4 and num_steps=10, we need 40 examples
n_train_examples = batch_size * num_steps
train_dataset = dataset.select(range(n_train_examples))

print(f"\nDataset: AG News")
print(f"Training examples: {len(train_dataset)}")
print(f"Examples per step: {batch_size}")
print(f"Total steps: {num_steps}")
print(f"\nSample:")
print(f"  Text: {train_dataset[0]['text'][:100]}...")

Loading AG News dataset...

Dataset: AG News
Training examples: 160
Examples per step: 4
Total steps: 40

Sample:
  Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...


In [20]:
# Tokenize dataset
texts = list(train_dataset["text"])
tokenized = tokenizer(
  texts,
  padding=True,
  truncation=True,
  max_length=128,
  return_tensors="pt",
)

input_ids = tokenized["input_ids"]
attention_mask = tokenized["attention_mask"]
labels = input_ids.clone()  # Causal LM: predict next token

print(f"Tokenized data:")
print(f"  Input IDs: {input_ids.shape}")
print(f"  Attention mask: {attention_mask.shape}")
print(f"  Labels: {labels.shape}")

Tokenized data:
  Input IDs: torch.Size([160, 116])
  Attention mask: torch.Size([160, 116])
  Labels: torch.Size([160, 116])


In [21]:
# Create PyTorch Dataset
from torch.utils.data import TensorDataset, DataLoader

# Create dataset from tokenized data
train_dataset_torch = TensorDataset(input_ids, attention_mask, labels)

# Create DataLoader with fixed batch size
# shuffle=True for better training (sample without replacement each epoch)
train_loader = DataLoader(
  train_dataset_torch,
  batch_size=batch_size,
  shuffle=True,
  drop_last=False,  # Keep all examples
)

print(f"\nDataLoader created:")
print(f"  Batch size: {batch_size}")
print(f"  Number of batches: {len(train_loader)}")
print(f"  Total examples: {len(train_dataset_torch)}")
print(f"  Shuffle: True (sample without replacement)")


DataLoader created:
  Batch size: 4
  Number of batches: 40
  Total examples: 160
  Shuffle: True (sample without replacement)


---

## Part 4: Converting to Functional Form

To use Opaque's optimizers, we need to convert the stateful model to functional form.

In [22]:
# Convert to functional form
print("Converting model to functional form...")

fmodel, trainable_params, frozen_params = make_functional(
  model,
  disable_autograd_tracking=True,
  partition_trainable=True,  # Separate trainable (LoRA) from frozen params
)

# Count parameters
trainable_count = sum(p.numel() for p in trainable_params.values())
frozen_count = sum(p.numel() for p in frozen_params.values())

print(f"\n✓ Functional conversion complete!")
print(f"  Trainable parameters: {trainable_count:,}")
print(f"  Frozen parameters: {frozen_count:,}")
print(f"  Trainable ratio: {trainable_count / (trainable_count + frozen_count):.2%}")

Converting model to functional form...

✓ Functional conversion complete!
  Trainable parameters: 294,912
  Frozen parameters: 124,439,808
  Trainable ratio: 0.24%


### Define Per-Example Loss Function

The loss function must operate on a single example for gradient clipping to work.

In [23]:
def per_example_loss(trainable, frozen, input_ids_single, mask_single, labels_single):
  """
  Compute loss for a single example.

  Args:
      trainable: Trainable parameters (LoRA)
      frozen: Frozen parameters (base model)
      input_ids_single: Input token IDs [seq_len]
      mask_single: Attention mask [seq_len]
      labels_single: Target labels [seq_len]

  Returns:
      Scalar loss
  """
  # Combine parameters
  all_params = {**frozen, **trainable}

  # Add batch dimension
  input_batch = input_ids_single.unsqueeze(0)  # [1, seq_len]
  mask_batch = mask_single.unsqueeze(0)  # [1, seq_len]
  labels_batch = labels_single.unsqueeze(0)  # [1, seq_len]

  # Forward pass
  outputs = fmodel(
    all_params,
    input_batch,
    attention_mask=mask_batch,
    labels=labels_batch,
  )

  return outputs.loss


print("✓ Per-example loss function defined")

✓ Per-example loss function defined


---

## Part 5: DP-SGD with DP-AdamW

### From SGD to Adam for DP Training

In Tutorial 03, we manually implemented DP-SGD with basic SGD updates. For production training, we want to use **Adam** (adaptive moments) which:
- Adapts learning rates per parameter
- Uses momentum for faster convergence
- Is standard for training neural networks

### DP-AdamW Algorithm

**Standard AdamW** (non-private):
```python
# 1. Compute gradient
g = compute_gradient(loss)

# 2. Update moments
m = β1 * m + (1 - β1) * g          # First moment (momentum)
v = β2 * v + (1 - β2) * g²         # Second moment (adaptive LR)

# 3. Update parameters
θ = θ - lr * m / (√v + ε) - lr * λ * θ  # AdamW with weight decay
```

**DP-AdamW** (with privacy):
```python
# 1. Compute per-example gradients and clip
for each example i:
    g_i = compute_gradient(loss_i)
    g_i = clip(g_i, max_norm=C)  # Clip to norm C

# 2. Sum and add noise
g = Σ g_i + N(0, σ² C² I)  # Gaussian noise scaled by clip norm

# 3. Update moments (same as standard Adam)
m = β1 * m + (1 - β1) * g
v = β2 * v + (1 - β2) * g²

# 4. Update parameters
θ = θ - lr * m / (√v + ε) - lr * λ * θ
```

**Key difference**: Only step 1-2 change for DP (clipping + noise). The Adam update logic (steps 3-4) stays the same!

---

In [24]:
# Setup clipped gradient function
clipped_grad_fn = clipped_grad(
  per_example_loss,
  argnums=0,  # Differentiate w.r.t. trainable params
  batch_argnums=(2, 3, 4),  # input_ids, mask, labels are batched
  l2_clip_norm=clip_norm,
  return_grad_norms=True,  # For debugging
  return_values=True,  # Monitor loss values
)

print(f"✓ Clipped gradient function created")
print(f"  Clip norm: {clip_norm}")

✓ Clipped gradient function created
  Clip norm: 0.1


In [26]:
# Create DP-AdamW optimizer
from opaque.optimizers import dp_adamw

init_fn, step_fn = dp_adamw(
  learning_rate=learning_rate,
  noise_multiplier=noise_multiplier,
  l2_clip_norm=clip_norm,
  sample_rate=sample_rate,
  target_delta=target_delta,
)

# Initialize optimizer state
opt_state = init_fn(trainable_params)

print(f"✓ DP-AdamW optimizer initialized!")
print(f"  Learning rate: {learning_rate}")
print(f"  Clip norm: {clip_norm}")
print(f"  Noise multiplier: {noise_multiplier}")

✓ DP-AdamW optimizer initialized!
  Learning rate: 0.0005
  Clip norm: 0.1
  Noise multiplier: 0.5


### Training Loop

Now let's train the model with our DP-AdamW optimizer:

**What happens in each step**:
1. **Compute clipped gradients**: Use `clipped_grad()` to get per-example gradients
2. **Optimizer step**: `dp_adamw` handles noise injection, parameter updates, and privacy tracking
3. **Track progress**: Monitor loss, gradient norms, and privacy cost (ε)

In [27]:
print("\nStarting DP-SGD training...")
print("=" * 80)

for step, (batch_input_ids, batch_attention_mask, batch_labels) in enumerate(train_loader):
  if step >= num_steps:
    break

  # Compute clipped gradients
  grads, aux = clipped_grad_fn(
    trainable_params,
    frozen_params,
    batch_input_ids,
    batch_attention_mask,
    batch_labels,
  )

  # Optimizer step: noise + updates + privacy tracking
  # Note: grads are already clipped and summed by clipped_grad_fn
  trainable_params, opt_state, metrics = step_fn(
    trainable_params,
    grads,
    opt_state,
  )

  # Compute statistics
  avg_loss = aux.values.mean().item()
  avg_grad_norm = aux.grad_norms.mean().item()

  # Print progress
  if (step + 1) % 5 == 0 or step == 0:
    print(f"\nStep {step + 1}/{num_steps}:")
    print(f"   Loss: {avg_loss:.4f}")
    print(f"   Avg grad norm: {avg_grad_norm:.2f}")
    print(f"   Privacy ε: {metrics['epsilon']:.3f}")

print("\n" + "=" * 80)
print("Training completed!")
print(f"Final privacy: ε = {metrics['epsilon']:.3f}, δ = {target_delta}")
print(f"Final loss: {avg_loss:.4f}")


Starting DP-SGD training...

Step 1/40:
   Loss: 8.3197
   Avg grad norm: 1.31
   Privacy ε: 4.396

Step 5/40:
   Loss: 6.8580
   Avg grad norm: 0.93
   Privacy ε: 5.207

Step 10/40:
   Loss: 8.5640
   Avg grad norm: 1.58
   Privacy ε: 5.630

Step 15/40:
   Loss: 8.3392
   Avg grad norm: 1.63
   Privacy ε: 5.893

Step 20/40:
   Loss: 7.4339
   Avg grad norm: 1.19
   Privacy ε: 6.131

Step 25/40:
   Loss: 7.5390
   Avg grad norm: 1.36
   Privacy ε: 6.359

Step 30/40:
   Loss: 8.3762
   Avg grad norm: 2.28
   Privacy ε: 6.502

Step 35/40:
   Loss: 8.6120
   Avg grad norm: 1.71
   Privacy ε: 6.644

Step 40/40:
   Loss: 8.1987
   Avg grad norm: 2.02
   Privacy ε: 6.787

Training completed!
Final privacy: ε = 6.787, δ = 1e-05
Final loss: 8.1987


---

## Part 6: Fixed Batch Sampling

### What is Fixed Batch Sampling?

In this tutorial, we used **fixed batch sampling** with PyTorch's DataLoader:

```python
train_loader = DataLoader(
    dataset,
    batch_size=4,      # Fixed size
    shuffle=True,      # Sample without replacement
    drop_last=False,
)
```

**Properties**:
- ✅ **Exact batch size**: Every batch has exactly `batch_size` examples (except possibly last)
- ✅ **Deterministic memory**: Predictable GPU memory usage
- ✅ **Standard practice**: What most practitioners use
- ✅ **RDP accounting**: Uses Rényi Differential Privacy for tight bounds

**Privacy Accounting**:
```python
accountant.step_fixed_batch(
    noise_multiplier=0.5,
    sample_rate=0.01,  # batch_size / dataset_size
    num_steps=1,
)
```

### Poisson Sampling (Tutorial 06)

**Poisson sampling** is an alternative where each example is included independently with probability `q`:

**Properties**:
- ⚠️ **Variable batch size**: Can be 0, very small, or very large
- ⚠️ **Memory issues**: Unpredictable memory usage
- ✅ **Tighter bounds**: Better privacy analysis (historically)
- ⚠️ **Requires microbatching**: To handle variable sizes

For now, **fixed batch sampling with DataLoader is the right choice**!

---

## Summary

### What We Built

A **production-ready DP-SGD training pipeline** for HuggingFace + LoRA:

1. ✅ Loaded pre-trained model (GPT-2) and applied LoRA with PEFT
2. ✅ Converted to functional form with `make_functional()`
3. ✅ Created PyTorch DataLoader for proper batch handling
4. ✅ **Used Opaque's `dp_adamw()` optimizer** for automatic DP training
5. ✅ Fixed batch sampling with RDP accounting
6. ✅ Clean, maintainable training loop

### Key Takeaways

✅ **DP-AdamW for Production DP Training**
- Combines Adam's adaptive learning rates with weight decay
- Automatically handles: clipping, noise injection, parameter updates, privacy tracking
- Clean, simple API: just 3 arguments to `step_fn()`
- Production-ready and less error-prone than manual implementation

✅ **LoRA + DP-SGD Works Well**
- LoRA: Small gradients (0.1-2.0) vs full fine-tuning (10-100+)
- Better signal-to-noise ratio with smaller clip norms
- Easier to tune hyperparameters

✅ **Fixed Batch Sampling is Practical**
- Predictable memory and compute
- Standard PyTorch DataLoader
- RDP accounting for tight privacy bounds
- Save Poisson sampling for Tutorial 05

✅ **Hyperparameters Matter**
- Clip norm: 0.1 for LoRA (matched to gradient scale)
- Learning rate: 5e-4 (higher than non-DP to overcome noise)
- Weight decay: 0.01 (explicit regularization)
- Batch size: 4-64 (as large as memory allows)
- Noise multiplier: 0.5-1.5 (calibrate for target ε)

---

## Complete Pattern

Here's the complete DP-SGD + LoRA pattern using Opaque:

```python
# 1. Load and apply LoRA
model = AutoModelForCausalLM.from_pretrained("gpt2")
model = get_peft_model(model, lora_config)

# 2. Convert to functional
fmodel, trainable, frozen = make_functional(
    model,
    partition_trainable=True,
)

# 3. Define per-example loss
def per_example_loss(trainable, frozen, inputs, mask, labels):
    all_params = {**frozen, **trainable}
    outputs = fmodel(all_params, inputs, attention_mask=mask, labels=labels)
    return outputs.loss

# 4. Setup gradient clipping
clipped_grad_fn = clipped_grad(
    per_example_loss,
    argnums=0,
    batch_argnums=(2, 3, 4),
    l2_clip_norm=0.1,
    return_grad_norms=True,
    return_values=True,
)

# 5. Create DP-AdamW optimizer
from opaque.optimizers import dp_adamw

init_fn, step_fn = dp_adamw(
    learning_rate=5e-4,
    weight_decay=0.01,
    l2_clip_norm=0.1,
    noise_multiplier=0.5,
    sample_rate=batch_size / len(dataset),
    target_delta=1e-5,
)
opt_state = init_fn(trainable)

# 6. Training loop
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

for batch_inputs, batch_mask, batch_labels in train_loader:
    # Compute clipped gradients
    grads, aux = clipped_grad_fn(trainable, frozen, batch_inputs, batch_mask, batch_labels)

    # Optimizer step (noise + updates + privacy)
    trainable, opt_state, metrics = step_fn(trainable, grads, opt_state)

    print(f"ε={metrics['epsilon']:.2f}, loss={aux.values.mean():.4f}")
```

**That's it!** Clean, production-ready DP-SGD for HuggingFace models.

---

## Comparison with Tutorial 03

| Aspect | Tutorial 03 (Manual) | Tutorial 04 (Production) |
|--------|---------------------|-------------------------|
| **Batching** | Manual indexing | PyTorch DataLoader ✅ |
| **Optimizer** | Manual SGD updates | DP-AdamW ✅ |
| **Code lines** | ~80 lines | ~25 lines ✅ |
| **Clipping** | Manual | Built into optimizer ✅ |
| **Noise** | Manual | Automatic ✅ |
| **Privacy** | Manual tracking | Automatic ✅ |
| **Purpose** | Educational | Production ✅ |

**Progression**:
- **Tutorial 03**: Learn the fundamentals (manual everything)
- **Tutorial 04**: Production patterns (DataLoader + optimizer API)
- **Tutorial 05**: Advanced techniques (Poisson + microbatching + adaptive clipping)

---

## What's Next?

**Tutorial 05** will cover:
1. **Poisson sampling**: Variable batch sizes for tighter privacy bounds
2. **Microbatching**: Handle variable batches efficiently
3. **Truncated Poisson**: Best of both worlds (variable + bounded)
4. **Adaptive clipping**: Automatically tune clip norm during training

**Other Extensions**:
1. **Multiple epochs**: Track privacy across full training runs
2. **Larger models**: Scale to Mistral-7B, LLaMA-2, etc.
3. **Evaluation**: Measure accuracy/perplexity vs privacy tradeoff
4. **Production deployment**: Export trained LoRA adapters

---

## Exercises

1. **Try different clip norms**: Train with 0.05, 0.1, 0.5 and compare results
2. **Vary privacy budget**: Train with ε=1.0, 3.0, 5.0 and observe utility
3. **Larger batch size**: Try batch_size=16, 32 (if memory allows)
4. **Different LoRA ranks**: Compare r=4, 8, 16 and observe gradient norms
5. **Weight decay ablation**: Train with weight_decay=0.0, 0.01, 0.1 and compare

---

🎉 **Congratulations!** You can now train HuggingFace models with differential privacy using production-ready patterns!

**Resources**:
- [LoRA Paper](https://arxiv.org/abs/2106.09685) (Hu et al., 2021)
- [DP-SGD Paper](https://arxiv.org/abs/1607.00133) (Abadi et al., 2016)
- [PEFT Documentation](https://huggingface.co/docs/peft/)
- [Opaque Documentation](https://github.com/evgri243/opaque)
- [Example: GPT-2 Basic Training](../../examples/gpt2_basic_training.py)

Questions? Open an issue on [GitHub](https://github.com/evgri243/opaque/issues).